# IAC-26 — Data Extraction & Labeling Notebook (Revised)
**Probabilistic Solar Eruption Forecasting for Radiation Risk Mitigation in Deep Space Missions**
IAC-26 · D5/IP/15 · Paper ID x111222 — Saatvik Sumanta · Kartikey Prasad · Anshika Ranjan · Austin George Francis

Revision of the *Original Dataset notebook* (`IAC_Research_Dataset - RAM efficient`) implementing the 10-phase
Dataset Making plan with the following **changes vs. the original**:

| # | Change | Where |
|---|--------|-------|
| 1 | GOES catalog **deduplication** on `(peak_time, goes_class)` + NOAA AR normalization to 5-digit form, applied *inside* `fetch_goes_flares()` before caching | Phase 7.1 |
| 2 | 2018–2019 zero M/X-flare gap **accepted as genuine deep solar minimum** — no backfill, test split unchanged | Phase 7.1 |
| 3 | Flare↔CME association window for `confined_flag` **widened to asymmetric −30 min / +120 min** (DONKI `time21_5` lags eruption onset due to coronagraph detection latency) | Phase 7.4 |
| 4 | DONKI `source_location` documented as **empty across all ~2,835 records** (API-side); `lat`/`lon` used for spatial work instead | Phase 7.2 |
| 5 | **Asymmetric stratified sampling** replaces flat stride-1 train materialization: positives ~full density, negatives stratified by `(year, activity_bucket)` at `TARGET_RATIO = 6:1`. **Val/test untouched** (prevalence-preserving, stride 4/6) | Phase 8-S (new) |
| 6 | Shards stored as **float16** (storage only — training must upcast to float32) | Phase 8 |

Expected output: **~79,300 windows** total (~29,800 train / 29,448 val / 20,035 test), **~395 GB** compressed at float16
(vs. 209,849 windows / ~1.5–2 TB at the original float32 stride-1 construction).


In [ ]:
# =====================================================================
# SETUP — Install dependencies
# =====================================================================
# zarr 2.x + numcodecs pinned to match the SDOML v2 store format.
!pip install -q zarr==2.12 numcodecs==0.10.2 s3fs dask scikit-image pandas requests pyyaml huggingface_hub tqdm sunpy pyarrow

In [ ]:
# =====================================================================
# SETUP — Imports & global configuration
# =====================================================================
import os
import gc
import json
import warnings
import logging
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import dask.array as da
import zarr
import s3fs
import requests
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, binary_dilation
from skimage.transform import downscale_local_mean
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("sdoml_dataset")

In [ ]:
# ---------------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------------
AIA_ZARR_PATH = "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2.zarr/"
HMI_ZARR_PATH = "s3://gov-nasa-hdrl-data1/contrib/fdl-sdoml/fdl-sdoml-v2/sdomlv2_hmi.zarr/"

YEARS = [str(y) for y in range(2010, 2021)]            # 2010–2020 inclusive

# Phase 2 — channel selection (7-channel working set: 4 AIA + 3 HMI)
AIA_CHANNELS = ["131A", "171A", "193A", "1600A"]
HMI_CHANNELS = ["Bx", "By", "Bz"]
RAW_CHANNELS = AIA_CHANNELS + HMI_CHANNELS

# Explicitly EXCLUDED channels (documented per plan Step 2.2):
#   94A   — poor quiet-sun SNR; marginal gain over 131A
#   211A  — largely redundant with 193A for eruption forecasting
#   304A  — chromospheric contamination dominates; unreliable filament detection
#   335A  — weakest coronal signal, poor SNR, minimal literature support
#   1700A, 4500A — photospheric; minimal coronal eruption sensitivity
#   EVE   — excluded entirely per project scope (AIA + HMI only)
EXCLUDED_CHANNELS = ["94A", "094A", "211A", "304A", "335A", "1700A", "4500A"]

# Phase 3 — alignment
HMI_CADENCE_MIN = 12
AIA_MATCH_TOL = pd.Timedelta(minutes=3)                 # |t_aia − t_hmi| < 3 min

# Phase 4 — cleaning
COSMIC_RAY_SIGMA = 5.0
DISAMBIG_SIGMA = 3.0
DISAMBIG_ROLL = 30                                       # rolling window (frames)
LIMB_RADIUS_FRAC = 0.98                                  # mask beyond 0.98 R_sun

# Phase 5 — derived features
MU0 = 4.0 * np.pi * 1e-7                                 # vacuum permeability
PIL_BTOTAL_THRESH = 150.0                                # Gauss, strong-field PIL
PIL_DILATE_PX = 5

# Phase 6 — normalisation
B_SAT = 1500.0                                           # Gauss, tanh saturation
ARCSINH_A_FRAC = 0.01                                    # a = train_median * 0.01
CLIP_SIGMA = 5.0                                         # for Jz / flux-rate tails

# Phase 7 — labels
FLARE_M_THRESH = 1e-5                                    # W/m2 (M1.0)
FLARE_X_THRESH = 1e-4                                    # W/m2 (X1.0)
CME_FAST_KMS = 500.0
FORECAST_HORIZONS_H = [6, 12, 24, 48]

# Phase 7.4 — WIDENED asymmetric flare<->CME association window (REVISED).
# Original +/-30 min under-associated by ~an order of magnitude (confined
# fractions 0.935 M / 0.940 X vs. the ~0.4 literature expectation): DONKI's
# `time21_5` marks coronagraph detection at 21.5 R_sun, which LAGS the true
# eruption onset. Asymmetric window: -30 min ... +120 min around flare peak.
CME_ASSOC_BEFORE = pd.Timedelta(minutes=30)              # CME may precede peak slightly
CME_ASSOC_AFTER = pd.Timedelta(minutes=120)              # detection latency allowance

# Phase 8 — windows
T_IN = 8                                                 # input frames (96 min @ 12 min)
TARGET_SIZE = 256                                        # downsample 512 -> 256
STRIDE = {"train": 1, "val": 4, "test": 6}               # index built at these strides;
                                                         # TRAIN materialization is then
                                                         # filtered by stratified sampling

# Phase 8-S — REVISED sampling strategy (train split ONLY; val/test untouched)
TARGET_RATIO = 6            # negatives : positives in materialized train set (tunable)
APPLY_M_THINNING = False    # optional stride-2 thinning of M-only positive clusters
SAMPLING_SEED = 42
ACTIVITY_SAMPLE_EVERY = 30  # alignment rows between activity-proxy samples (30 rows = 6 h)
ACTIVITY_MAX_PER_YEAR = None  # int cap for demo runs; None = full

# Phase 8 — REVISED storage dtype. float16 halves shard size (~395 GB total).
# *** STORAGE FORMAT ONLY *** — the training notebook MUST upcast to float32
# (or run a mixed-precision policy) immediately after loading: NLL / logvar
# loss terms are numerically unstable computed directly in fp16.
STORE_DTYPE = np.float16

# Phase 9 — chronological splits (UNCHANGED — incl. 2018–2020 test boundary)
SPLITS = {
    "train": (2010, 2014),   # Cycle 24 rising + maximum
    "val":   (2015, 2017),   # Cycle 24 declining
    "test":  (2018, 2020),   # Cycle 24 minimum + Cycle 25 rising
}

OUT_DIR = "./sdoml_dataset_out"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Working channel set ({len(RAW_CHANNELS)}): {RAW_CHANNELS}")
print(f"Excluded: {EXCLUDED_CHANNELS} + all EVE data")
print(f"Sampling: TARGET_RATIO={TARGET_RATIO}:1 (train only) | storage dtype={STORE_DTYPE.__name__}")
print(f"CME association window: -{CME_ASSOC_BEFORE} ... +{CME_ASSOC_AFTER} around flare peak")

In [ ]:
# =====================================================================
# PHASE 1 / Step 1.1 - Connect to AWS S3 anonymously and open stores
# =====================================================================
# Bounded LRU chunk cache: max_size=None would retain every S3 chunk ever read
# and OOM-kill the kernel on a full-archive run. Cap keeps the resident set flat.
ZARR_CACHE_BYTES = 1 * 1024**3        # 1 GiB resident chunk cache per store


def s3_connection(path: str) -> s3fs.S3Map:
    """Anonymous S3Map - no AWS credentials needed."""
    return s3fs.S3Map(root=path, s3=s3fs.S3FileSystem(anon=True), check=False)


def load_zarr_from_s3(path: str, cache_max_single_size: Optional[int] = ZARR_CACHE_BYTES):
    """Open a Zarr group from S3 behind a bounded LRU chunk cache (metadata only)."""
    return zarr.open(
        zarr.LRUStoreCache(store=s3_connection(path), max_size=cache_max_single_size),
        mode="r",
    )


log.info("Opening AIA full store (sdomlv2.zarr, 2010-2020) ...")
aia_root = load_zarr_from_s3(AIA_ZARR_PATH)

log.info("Opening HMI full store (sdomlv2_hmi.zarr, 2010-2020) ...")
hmi_root = load_zarr_from_s3(HMI_ZARR_PATH)

log.info("Stores connected (metadata only - no pixel data downloaded yet).")

In [ ]:
# =====================================================================
# PHASE 1 / Step 1.2 — Enumerate and inventory year groups
# =====================================================================
def build_inventory(root: zarr.Group, wanted_channels: List[str], label: str) -> Dict[str, Dict[str, tuple]]:
    """
    Return {year: {channel: shape}} for the requested channels only.
    Logs missing channels per year as warnings — never silently skips.
    """
    inv: Dict[str, Dict[str, tuple]] = {}
    year_keys = sorted(k for k in root.keys() if k.isdigit() and k in YEARS)
    for yr in year_keys:
        grp = root[yr]
        if not isinstance(grp, zarr.Group):
            continue
        inv[yr] = {}
        available = set(grp.keys())
        for ch in wanted_channels:
            if ch in available:
                arr = grp[ch]
                inv[yr][ch] = arr.shape          # (T, 512, 512); T differs per channel
            else:
                log.warning(f"{label} {yr}: channel {ch} MISSING from store")
    return inv


aia_inv = build_inventory(aia_root, AIA_CHANNELS, "AIA")
hmi_inv = build_inventory(hmi_root, HMI_CHANNELS, "HMI")

print(f"{'Year':<6} | " + " | ".join(f"{c:>10}" for c in RAW_CHANNELS))
print("-" * (8 + 13 * len(RAW_CHANNELS)))
for yr in YEARS:
    row = [f"{yr:<6}"]
    for ch in AIA_CHANNELS:
        row.append(f"{aia_inv.get(yr, {}).get(ch, ('-',))[0]:>10}")
    for ch in HMI_CHANNELS:
        row.append(f"{hmi_inv.get(yr, {}).get(ch, ('-',))[0]:>10}")
    print(" | ".join(row))

In [ ]:
# =====================================================================
# PHASE 1 / Step 1.3 - Wrap all arrays as lazy Dask arrays
# =====================================================================
# Pure computation graph — nothing downloaded. Native zarr chunking is kept so
# single-frame fetches read the minimum; full-archive reductions rechunk on the
# fly to REDUCE_TIME_CHUNK for scheduler throughput.
REDUCE_TIME_CHUNK = 64

lazy_store: Dict[str, Dict[str, da.Array]] = {}
for yr in YEARS:
    lazy_store[yr] = {}
    for ch in AIA_CHANNELS:
        if ch in aia_inv.get(yr, {}):
            lazy_store[yr][ch] = da.from_zarr(aia_root[yr][ch])
    for ch in HMI_CHANNELS:
        if ch in hmi_inv.get(yr, {}):
            lazy_store[yr][ch] = da.from_zarr(hmi_root[yr][ch])

n_arrays = sum(len(v) for v in lazy_store.values())
print(f"Lazy store built: {n_arrays} dask arrays across {len(lazy_store)} years")

In [ ]:
# =====================================================================
# PHASE 3 / Step 3.1 — Load timestamp indices from Zarr metadata
# =====================================================================
# Frames inside the Zarr arrays are NOT guaranteed time-ordered: sort by T_OBS
# and carry the original array index through the alignment table.
def load_timestamps(root: zarr.Group, year: str, channel: str) -> pd.DatetimeIndex:
    attrs = root[year][channel].attrs
    for t_key in ("T_OBS", "T_REC", "Time", "time"):
        if t_key in attrs:
            val = np.array(attrs[t_key])
            if val.ndim > 0 and len(val) > 1:
                raw = val
                break
    else:
        raise KeyError(f"No per-frame time array in {year}/{channel}")

    # Handle JSOC format: '2010.05.01_00:12:04_TAI' -> '2010-05-01 00:12:04'
    cleaned = (pd.Series(raw)
               .str.replace("_TAI", "", regex=False)
               .str.replace("_UTC", "", regex=False)
               .str.rstrip("Z")
               .str.replace(r"^(\d{4})\.(\d{2})\.(\d{2})_", r"\1-\2-\3 ", regex=True))
    return pd.DatetimeIndex(pd.to_datetime(cleaned, errors="coerce"))


# timestamps[year][channel] -> (sorted DatetimeIndex, argsort index into raw zarr array)
timestamps: Dict[str, Dict[str, Tuple[pd.DatetimeIndex, np.ndarray]]] = {}
for yr in tqdm(YEARS, desc="Loading timestamps"):
    timestamps[yr] = {}
    for ch in RAW_CHANNELS:
        if ch not in lazy_store.get(yr, {}):
            continue
        root = aia_root if ch in AIA_CHANNELS else hmi_root
        t_raw = load_timestamps(root, yr, ch)
        order = np.argsort(t_raw.values)          # map sorted position -> raw zarr index
        timestamps[yr][ch] = (t_raw[order], order)

# Cadence sanity check (AIA ~6 min, HMI ~12 min)
for ch in ["171A", "Bz"]:
    for yr in YEARS:
        if ch in timestamps.get(yr, {}):
            ts, _ = timestamps[yr][ch]
            med_dt = pd.Series(ts).diff().median()
            print(f"{ch} {yr}: {len(ts)} frames, median cadence = {med_dt}")
            break

In [ ]:
# =====================================================================
# PHASE 3 / Step 3.2 — Build unified 12-minute timeline anchored on HMI
# =====================================================================
def nearest_match(anchor: pd.DatetimeIndex, candidates: pd.DatetimeIndex,
                  tol: pd.Timedelta) -> np.ndarray:
    """
    For each anchor time, return the index (into `candidates`, sorted) of the
    nearest candidate within `tol`, else -1.
    """
    pos = candidates.searchsorted(anchor)                    # insertion points
    pos = np.clip(pos, 1, len(candidates) - 1)
    left, right = candidates[pos - 1], candidates[pos]
    use_left = (anchor.values - left.values) <= (right.values - anchor.values)
    best = np.where(use_left, pos - 1, pos)
    dt = np.abs(anchor.values - candidates.values[best])
    best = np.where(dt < np.timedelta64(int(tol.total_seconds()), "s"), best, -1)
    return best.astype(np.int64)


def build_alignment_table(year: str) -> Optional[pd.DataFrame]:
    """
    One row per HMI timestamp. Columns: time, idx_<ch> (raw zarr index, -1 = no
    match), fully_matched (all 4 AIA + 3 HMI present within tolerance).
    """
    if not all(ch in timestamps.get(year, {}) for ch in HMI_CHANNELS):
        log.warning(f"{year}: HMI channels incomplete — year skipped")
        return None

    t_anchor, _ = timestamps[year]["Bz"]
    table = pd.DataFrame({"time": t_anchor})

    for ch in RAW_CHANNELS:
        if ch not in timestamps[year]:
            table[f"idx_{ch}"] = -1
            continue
        ts_sorted, order = timestamps[year][ch]
        m = nearest_match(t_anchor, ts_sorted, AIA_MATCH_TOL)
        raw_idx = np.where(m >= 0, order[np.clip(m, 0, None)], -1)
        table[f"idx_{ch}"] = raw_idx

    idx_cols = [f"idx_{ch}" for ch in RAW_CHANNELS]
    table["fully_matched"] = (table[idx_cols] >= 0).all(axis=1)
    table["partially_missing"] = ~table["fully_matched"]
    return table


align: Dict[str, pd.DataFrame] = {}
for yr in tqdm(YEARS, desc="Aligning years"):
    t = build_alignment_table(yr)
    if t is not None:
        align[yr] = t

In [ ]:
# =====================================================================
# PHASE 3 / Step 3.3 — Log coverage statistics per year
# =====================================================================
coverage_rows = []
for yr, t in align.items():
    n = len(t)
    full = int(t["fully_matched"].sum())
    aia_cols = [f"idx_{c}" for c in AIA_CHANNELS]
    partial = int(((t[aia_cols] >= 0).any(axis=1) & ~t["fully_matched"]).sum())
    dropped = n - full - partial
    coverage_rows.append({
        "year": yr,
        "hmi_timestamps": n,
        "pct_fully_matched": round(100 * full / n, 2),
        "pct_partial": round(100 * partial / n, 2),
        "pct_dropped": round(100 * dropped / n, 2),
    })

coverage_df = pd.DataFrame(coverage_rows).set_index("year")
print("=== TRUE EFFECTIVE COVERAGE (before window construction) ===")
print(coverage_df.to_string())
coverage_df.to_csv(os.path.join(OUT_DIR, "coverage_stats.csv"))

In [ ]:
# =====================================================================
# PHASE 4 / Steps 4.1, 4.2, 4.5 — Per-frame cleaning + validity mask
# =====================================================================
def remove_cosmic_rays(frame: np.ndarray, n_sigma: float = COSMIC_RAY_SIGMA
                       ) -> Tuple[np.ndarray, float]:
    """3x3 median filter; replace pixels deviating > n_sigma*(local sigma) from local median."""
    med = median_filter(frame, size=3)
    resid = frame - med
    sigma = np.std(resid)
    if sigma == 0:
        return frame, 0.0
    bad = np.abs(resid) > n_sigma * sigma
    cleaned = np.where(bad, med, frame)
    return cleaned, float(bad.mean())


def make_limb_mask(size: int = 512, r_frac: float = LIMB_RADIUS_FRAC) -> np.ndarray:
    """Boolean mask: True inside r_frac * R_sun (disk centred, R_sun = size/2)."""
    yy, xx = np.mgrid[0:size, 0:size]
    c = (size - 1) / 2.0
    r = np.sqrt((xx - c) ** 2 + (yy - c) ** 2)
    return r <= r_frac * (size / 2.0)


LIMB_MASK_512 = make_limb_mask(512)
print(f"Limb mask: {LIMB_MASK_512.mean()*100:.1f}% of pixels retained "
      f"(inside {LIMB_RADIUS_FRAC} R_sun)")


def apply_limb_mask(frame: np.ndarray) -> np.ndarray:
    """Zero all pixels beyond 0.98 R_sun — applied before normalisation/features."""
    return np.where(LIMB_MASK_512, frame, 0.0)


def clean_frame(frame: np.ndarray) -> Tuple[np.ndarray, float]:
    """Full per-frame cleaning: cosmic rays then limb mask."""
    cleaned, frac = remove_cosmic_rays(frame.astype(np.float32))
    return apply_limb_mask(cleaned), frac


def build_valid_mask(table: pd.DataFrame) -> pd.DataFrame:
    """
    Per-timestamp validity columns:
      valid      — usable as-is (all channels matched)
      interp_ok  — isolated single-frame gap, flanking frames exist (interpolable)
      unusable   — >= 2 consecutive gaps, or gap at sequence edge
    """
    missing = ~table["fully_matched"].values
    n = len(table)
    valid = ~missing
    interp_ok = np.zeros(n, dtype=bool)
    for i in np.where(missing)[0]:
        prev_ok = i > 0 and not missing[i - 1]
        next_ok = i < n - 1 and not missing[i + 1]
        if prev_ok and next_ok:                 # isolated single-frame gap
            interp_ok[i] = True
    out = table.copy()
    out["valid"] = valid
    out["interp_ok"] = interp_ok
    out["unusable"] = missing & ~interp_ok
    return out


for yr in align:
    align[yr] = build_valid_mask(align[yr])

summary = pd.DataFrame({
    yr: {
        "valid": int(t["valid"].sum()),
        "interp_ok": int(t["interp_ok"].sum()),
        "unusable": int(t["unusable"].sum()),
    } for yr, t in align.items()
}).T
print(summary.to_string())

In [ ]:
# =====================================================================
# PHASE 4 / Step 4.3 - HMI 180-degree disambiguation artifact detection
# =====================================================================
def detect_disambiguation_jumps(year: str) -> np.ndarray:
    """Return boolean array (per HMI timestamp) of suspected disambiguation jumps."""
    t = align[year]
    bx = lazy_store[year]["Bx"].rechunk({0: REDUCE_TIME_CHUNK})
    abs_bx_sum = da.nansum(da.fabs(bx), axis=(1, 2)).compute()

    idx = t["idx_Bx"].values
    series = np.where(idx >= 0, abs_bx_sum[np.clip(idx, 0, None)], np.nan)
    diff = np.abs(np.diff(series, prepend=series[0]))

    roll_std = pd.Series(diff).rolling(DISAMBIG_ROLL, min_periods=5).std().values
    jumps = diff > DISAMBIG_SIGMA * np.where(np.isnan(roll_std), np.inf, roll_std)
    return np.nan_to_num(jumps, nan=False).astype(bool)


# Full-archive scan is expensive (one pass over every HMI Bx chunk).
RUN_DISAMBIG_SCAN = False    # True for the production run

for yr in align:
    if RUN_DISAMBIG_SCAN:
        jumps = detect_disambiguation_jumps(yr)
        align[yr]["disambig_jump"] = jumps
        align[yr].loc[jumps, "unusable"] = True
        align[yr].loc[jumps, "valid"] = False
        log.info(f"{yr}: {jumps.sum()} disambiguation jumps flagged")
    else:
        align[yr]["disambig_jump"] = False

print("Disambiguation scan:", "DONE" if RUN_DISAMBIG_SCAN else
      "SKIPPED (set RUN_DISAMBIG_SCAN=True for production)")

In [ ]:
# =====================================================================
# PHASE 4 / Step 4.4 - AIA degradation verification (monthly 95th percentile)
# =====================================================================
def monthly_p95(year: str, channel: str, sample_stride: int = 20) -> pd.Series:
    """Monthly 95th-percentile of per-frame mean intensity (cheap degradation proxy)."""
    attrs = dict(aia_root[year][channel].attrs)            # metadata only
    ts, order = timestamps[year][channel]
    if "DATAMEAN" in attrs:
        stat = np.array(attrs["DATAMEAN"], dtype=float)[order]
    else:
        sub = lazy_store[year][channel][::sample_stride].astype(np.float32)
        stat_sub = da.nanmean(sub, axis=(1, 2)).compute()
        stat = np.interp(np.arange(len(ts)),
                         np.arange(0, len(ts), sample_stride)[:len(stat_sub)], stat_sub)
    s = pd.Series(stat, index=ts)
    return s.resample("MS").quantile(0.95)


RUN_DEGRADATION_CHECK = True
degradation_correction: Dict[str, Dict[str, float]] = {ch: {} for ch in AIA_CHANNELS}

if RUN_DEGRADATION_CHECK:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, ch in zip(axes.flat, AIA_CHANNELS):
        parts = [monthly_p95(yr, ch) for yr in YEARS if ch in timestamps.get(yr, {})]
        if not parts:
            continue
        series = pd.concat(parts).sort_index()
        ax.plot(series.index, series.values, lw=1)
        ax.set_title(f"AIA {ch} — monthly p95 intensity"); ax.grid(alpha=0.3)

        x = (series.index - series.index[0]).days.values.astype(float)
        y = np.log(np.clip(series.values, 1e-3, None))
        slope, intercept = np.polyfit(x[~np.isnan(y)], y[~np.isnan(y)], 1)
        trend_per_year = float(np.exp(slope * 365.25))
        if trend_per_year < 0.97:   # > 3%/yr systematic decline -> correct
            for i, yr in enumerate(YEARS):
                degradation_correction[ch][yr] = float(trend_per_year ** (-i))
            log.warning(f"AIA {ch}: residual degradation {100*(1-trend_per_year):.1f}%/yr "
                        f"-> per-year multiplicative correction stored")
        else:
            log.info(f"AIA {ch}: no significant residual degradation "
                     f"(trend {100*(1-trend_per_year):+.2f}%/yr) — SDOML correction OK")
    plt.tight_layout(); plt.show()


def degradation_factor(channel: str, year: str) -> float:
    """Multiplicative correction (1.0 when SDOML's own correction is sufficient)."""
    return degradation_correction.get(channel, {}).get(year, 1.0)

In [ ]:
# =====================================================================
# PHASE 5 — Derived physical feature maps (Jz, PIL, shear, flux, grad171)
# =====================================================================
# | # | Feature                     | Source                                       | Type          |
# |---|-----------------------------|----------------------------------------------|---------------|
# | 8 | Jz vertical current density | Bx, By central differences                   | map channel   |
# | 9 | PIL binary mask             | Bz zero-crossing, |B| > 150 G, 5-px dilation | map channel   |
# | 10| 171A gradient magnitude     | |grad I_171|                                 | map channel   |
# | – | PIL length                  | count of PIL pixels                          | scalar/frame  |
# | – | Mean shear angle at PIL     | (Bx,By) vs PIL-perpendicular                 | scalar/frame  |
# | – | Unsigned flux & dPhi/dt     | sum|Bz|*pixel_area, 12-min finite diff       | scalar series |

def compute_jz(bx: np.ndarray, by: np.ndarray, pixel_size_m: float = 0.504e6) -> np.ndarray:
    """Jz from central finite differences; inputs must already be limb-masked."""
    dby_dx = np.gradient(by, pixel_size_m, axis=1)
    dbx_dy = np.gradient(bx, pixel_size_m, axis=0)
    jz = (dby_dx - dbx_dy) / MU0
    return apply_limb_mask(jz).astype(np.float32)


def compute_pil_mask(bx: np.ndarray, by: np.ndarray, bz: np.ndarray
                     ) -> Tuple[np.ndarray, int]:
    """Binary PIL mask: Bz zero-crossing where |B_total| > 150 G, dilated 5 px."""
    b_total = np.sqrt(bx ** 2 + by ** 2 + bz ** 2)
    strong = b_total > PIL_BTOTAL_THRESH
    sign = np.sign(bz)
    zc = np.zeros_like(bz, dtype=bool)
    zc[:-1, :] |= (sign[:-1, :] * sign[1:, :]) < 0
    zc[:, :-1] |= (sign[:, :-1] * sign[:, 1:]) < 0
    pil_core = zc & strong & LIMB_MASK_512
    pil_len = int(pil_core.sum())
    mask = binary_dilation(pil_core, iterations=PIL_DILATE_PX)
    return mask.astype(np.float32), pil_len


def compute_shear_angle(bx: np.ndarray, by: np.ndarray, pil_core_mask: np.ndarray) -> float:
    """Mean angle (deg) between observed horizontal field and PIL-perpendicular proxy."""
    if pil_core_mask.sum() < 10:
        return 0.0
    gy, gx = np.gradient(pil_core_mask.astype(float))
    pot_x, pot_y = gx, gy
    norm_pot = np.hypot(pot_x, pot_y)
    norm_obs = np.hypot(bx, by)
    sel = (pil_core_mask > 0) & (norm_pot > 0) & (norm_obs > 0)
    if sel.sum() == 0:
        return 0.0
    cosang = np.clip(
        (bx[sel] * pot_x[sel] + by[sel] * pot_y[sel]) / (norm_obs[sel] * norm_pot[sel]),
        -1.0, 1.0)
    ang = np.degrees(np.arccos(np.abs(cosang)))    # fold to [0, 90]
    return float(np.mean(ang))


def compute_unsigned_flux(bz: np.ndarray, pixel_area_cm2: float = (0.504e8) ** 2) -> float:
    """Phi = sum |Bz| * pixel_area  [Mx]; limb-masked input expected."""
    return float(np.sum(np.abs(bz)) * pixel_area_cm2)


def compute_gradient_map(i171: np.ndarray) -> np.ndarray:
    """|grad I_171| — highlights loop boundaries and brightening fronts."""
    gy, gx = np.gradient(i171.astype(np.float32))
    return apply_limb_mask(np.hypot(gx, gy)).astype(np.float32)


print("Derived-feature functions ready: Jz, PIL mask, shear angle, unsigned flux, 171A gradient")

In [ ]:
# =====================================================================
# PHASE 5 + 8.5 - Frame assembly: cleaned 10-channel stack at 256x256
# =====================================================================
CHANNEL_ORDER = AIA_CHANNELS + HMI_CHANNELS + ["Jz", "PIL", "grad171"]   # 10 channels
N_CHANNELS = len(CHANNEL_ORDER)


def downsample_area(frame: np.ndarray, factor: int = 512 // TARGET_SIZE) -> np.ndarray:
    """Area-average 512->256. Preserves integrated flux (NOT bilinear/bicubic)."""
    return downscale_local_mean(frame, (factor, factor)).astype(np.float32)


def fetch_raw_frame(year: str, channel: str, raw_idx: int) -> np.ndarray:
    """One raw 512x512 frame via the lazy Dask graph (bounded LRU cache behind it)."""
    return np.asarray(lazy_store[year][channel][int(raw_idx)].compute()).astype(np.float32)


def fetch_raw_frames(year: str, channel: str, raw_indices) -> np.ndarray:
    """Fetch several frames of one channel through a SINGLE Dask graph / one compute."""
    idx = np.asarray(raw_indices, dtype=np.int64)
    if idx.size == 0:
        return np.empty((0, 512, 512), dtype=np.float32)
    return np.asarray(lazy_store[year][channel][idx].compute()).astype(np.float32)


def _clean_and_correct(year: str, channel: str, frame: np.ndarray) -> Tuple[np.ndarray, float]:
    """Per-frame cleaning (cosmic ray + limb mask) plus AIA degradation correction."""
    frame, frac = clean_frame(frame)                         # Steps 4.1 + 4.5
    if channel in AIA_CHANNELS:
        frame = frame * degradation_factor(channel, year)    # Step 4.4
    return frame, frac


def assemble_from_cleaned(year: str, cleaned512: Dict[str, np.ndarray],
                          cr_fracs: Dict[str, float]) -> Tuple[np.ndarray, Dict[str, float]]:
    """Derived features + 256x256 stack from already-cleaned 512x512 raw channels."""
    bx, by, bz = cleaned512["Bx"], cleaned512["By"], cleaned512["Bz"]

    jz = compute_jz(bx, by)
    pil_mask, pil_len = compute_pil_mask(bx, by, bz)
    grad171 = compute_gradient_map(cleaned512["171A"])

    scalars = {
        "pil_length_px": pil_len,
        "shear_angle_deg": compute_shear_angle(bx, by, pil_mask),
        "unsigned_flux_Mx": compute_unsigned_flux(bz),
        "cr_frac_mean": float(np.mean(list(cr_fracs.values()))),
    }

    stack = np.stack(
        [downsample_area(cleaned512[ch]) for ch in RAW_CHANNELS]
        + [downsample_area(jz), downsample_area(pil_mask), downsample_area(grad171)],
        axis=0,
    )  # (10, 256, 256)
    return stack, scalars


def assemble_timestep(year: str, row: pd.Series) -> Tuple[np.ndarray, Dict[str, float]]:
    """Full 10-channel (256,256) physical-units stack for one aligned timestamp."""
    cleaned512: Dict[str, np.ndarray] = {}
    cr_fracs: Dict[str, float] = {}
    for ch in RAW_CHANNELS:
        idx = int(row[f"idx_{ch}"])
        if idx < 0:
            raise ValueError(f"assemble_timestep called on unmatched channel {ch}")
        frame, frac = _clean_and_correct(year, ch, fetch_raw_frame(year, ch, idx))
        cleaned512[ch] = frame
        cr_fracs[ch] = frac
    return assemble_from_cleaned(year, cleaned512, cr_fracs)


print(f"Channel contract ({N_CHANNELS}): {CHANNEL_ORDER}")

In [ ]:
# =====================================================================
# PHASE 6 / Steps 6.1–6.3 — Fit scalers on TRAINING SPLIT ONLY (2010–2014)
# =====================================================================
TRAIN_YEARS = [str(y) for y in range(SPLITS["train"][0], SPLITS["train"][1] + 1)]
SCALER_SAMPLE_PER_YEAR = 50          # aligned timestamps sampled per train year


def fit_scalers(sample_per_year: int = SCALER_SAMPLE_PER_YEAR) -> Dict[str, dict]:
    """
    Sample valid aligned timestamps uniformly across 2010–2014, assemble cleaned
    10-channel stacks, and fit per-channel scalers.
    Leakage guarantee: only TRAIN_YEARS rows are ever touched here (Step 9.2).
    """
    samples = {ch: [] for ch in CHANNEL_ORDER}
    flux_vals = []
    for yr in TRAIN_YEARS:
        t = align.get(yr)
        if t is None:
            continue
        valid_rows = t[t["valid"]]
        if len(valid_rows) == 0:
            continue
        pick = valid_rows.iloc[np.linspace(0, len(valid_rows) - 1,
                                           min(sample_per_year, len(valid_rows)),
                                           dtype=int)]
        prev_flux = None
        for _, row in tqdm(pick.iterrows(), total=len(pick), desc=f"scaler sample {yr}", leave=False):
            try:
                stack, scal = assemble_timestep(yr, row)
            except Exception as e:
                log.warning(f"scaler sample failed @ {row['time']}: {e}")
                continue
            for ci, ch in enumerate(CHANNEL_ORDER):
                samples[ch].append(stack[ci].ravel()[::37])      # sparse pixel sample
            if prev_flux is not None:
                flux_vals.append((scal["unsigned_flux_Mx"] - prev_flux) / (HMI_CADENCE_MIN * 60))
            prev_flux = scal["unsigned_flux_Mx"]

    scalers: Dict[str, dict] = {}
    for ch in CHANNEL_ORDER:
        x = np.concatenate(samples[ch]) if samples[ch] else np.array([0.0, 1.0])
        if ch in AIA_CHANNELS:
            a = max(float(np.median(x[x > 0])) * ARCSINH_A_FRAC, 1e-3)
            xs = np.arcsinh(x / a)
            scalers[ch] = {"kind": "arcsinh", "a": a,
                           "mean": float(xs.mean()), "std": float(xs.std() + 1e-8)}
        elif ch in HMI_CHANNELS:
            xs = np.tanh(x / B_SAT)
            scalers[ch] = {"kind": "tanh", "b_sat": B_SAT,
                           "mean": float(xs.mean()), "std": float(xs.std() + 1e-8)}
        elif ch == "Jz":
            mu, sd = float(x.mean()), float(x.std() + 1e-8)
            xc = np.clip(x, mu - CLIP_SIGMA * sd, mu + CLIP_SIGMA * sd)
            scalers[ch] = {"kind": "clip_z", "clip": CLIP_SIGMA,
                           "mean": float(xc.mean()), "std": float(xc.std() + 1e-8)}
        elif ch == "PIL":
            scalers[ch] = {"kind": "binary"}                     # no normalisation
        elif ch == "grad171":
            scalers[ch] = {"kind": "z", "mean": float(x.mean()),
                           "std": float(x.std() + 1e-8)}

    fr = np.array(flux_vals) if flux_vals else np.array([0.0, 1.0])
    mu, sd = float(fr.mean()), float(fr.std() + 1e-8)
    fr_c = np.clip(fr, mu - CLIP_SIGMA * sd, mu + CLIP_SIGMA * sd)
    scalers["flux_rate"] = {"kind": "clip_z", "clip": CLIP_SIGMA,
                            "mean": float(fr_c.mean()), "std": float(fr_c.std() + 1e-8)}
    scalers["shear_angle"] = {"kind": "div", "div": 90.0}
    return scalers


RUN_SCALER_FIT = True
if RUN_SCALER_FIT:
    scalers = fit_scalers()
    with open(os.path.join(OUT_DIR, "scalers.json"), "w") as f:
        json.dump(scalers, f, indent=2)
    print(json.dumps(scalers, indent=2)[:1200])
else:
    with open(os.path.join(OUT_DIR, "scalers.json")) as f:
        scalers = json.load(f)

In [ ]:
# =====================================================================
# PHASE 6 — Apply-normalisation function + Step 6.4 Surya scaler alignment
# =====================================================================
def normalise_stack(stack: np.ndarray, scaler_set: Dict[str, dict]) -> np.ndarray:
    """Apply per-channel normalisation to a (10, 256, 256) physical-units stack."""
    out = np.empty_like(stack, dtype=np.float32)
    for ci, ch in enumerate(CHANNEL_ORDER):
        s, x = scaler_set[ch], stack[ci]
        if s["kind"] == "arcsinh":
            x = np.arcsinh(x / s["a"]);            x = (x - s["mean"]) / s["std"]
        elif s["kind"] == "tanh":
            x = np.tanh(x / s["b_sat"]);           x = (x - s["mean"]) / s["std"]
        elif s["kind"] == "clip_z":
            lo = s["mean"] - s["clip"] * s["std"]; hi = s["mean"] + s["clip"] * s["std"]
            x = np.clip(x, lo, hi);                x = (x - s["mean"]) / s["std"]
        elif s["kind"] == "z":
            x = (x - s["mean"]) / s["std"]
        elif s["kind"] == "binary":
            pass                                    # PIL mask stays {0,1}
        out[ci] = x
    return out


# Step 6.4 — Surya scaler alignment (Models D & F input variants only).
# Derived channels (Jz, PIL, grad171) keep our own scalers — Surya has no equivalent.
USE_SURYA_SCALERS = False     # set True when building inputs for Models D / F

if USE_SURYA_SCALERS:
    from huggingface_hub import hf_hub_download
    import yaml
    p = hf_hub_download(repo_id="nasa-ibm-ai4science/Surya-1.0", filename="scalers.yaml")
    with open(p) as f:
        surya = yaml.safe_load(f)
    SURYA_KEY = {"131A": "aia131", "171A": "aia171", "193A": "aia193",
                 "1600A": "aia1600", "Bx": "hmi_bx", "By": "hmi_by", "Bz": "hmi_bz"}
    scalers_surya = dict(scalers)
    for ch, key in SURYA_KEY.items():
        if key in surya:
            scalers_surya[ch] = {**surya[key], "kind": surya[key].get("kind", "arcsinh")}
            log.info(f"{ch}: replaced with Surya scaler ({key})")
        else:
            log.warning(f"{ch}: '{key}' not found in Surya scalers.yaml — keeping own")
    with open(os.path.join(OUT_DIR, "scalers_surya.json"), "w") as f:
        json.dump(scalers_surya, f, indent=2)

print("Normalisation ready. Surya alignment:", "ON" if USE_SURYA_SCALERS else "OFF (own scalers)")

## Phase 7 — Label Generation (REVISED catalog cleaning)

Both catalogs are produced **at runtime** by `fetch_goes_flares()` (HEK query) and `fetch_donki_cmes()`
(DONKI CMEAnalysis API), then disk-cached. The fixes below are applied **inside those functions,
immediately before `df.to_csv(cache)`**, so the corrected catalog is what gets cached and reused.

1. **GOES deduplication** on `(peak_time, goes_class)` — HEK aggregates multiple reporting pipelines, so the
   same physical flare can come back twice with inconsistently formatted `ar_noaanum` (`1121` vs `11121`).
   All `noaa_ar` values are normalized to well-formed 5-digit NOAA numbering; before/after row counts are logged.
2. **2018–2019 zero M/X-flare gap: accepted as-is.** Genuine deep-solar-minimum behavior, not a pull failure.
   No backfill, no re-verification; 2018–2020 stays in the test split exactly as planned.
3. **`source_location` is empty across every DONKI CME record (~2,835)** — the API returns no `sourceLocation`
   value for this period. Not an extraction bug; do not require it. Use `lat`/`lon` for spatial stratification.
4. M-class and X-class remain **separate label columns** throughout.


In [ ]:
# =====================================================================
# PHASE 7 / Step 7.1 — Fetch GOES flare catalog (M and X, 2010–2020) [REVISED]
# =====================================================================
def _normalize_noaa_ar(v) -> int:
    """
    Normalize NOAA active-region numbers to well-formed 5-digit form.
    NOAA numbering rolled past 10000 in 2002, so every 2010–2020 AR is 5-digit
    (11xxx–12xxx); 3–4 digit values from HEK (e.g. 1121) are truncated forms of
    the same region (11121). Missing/invalid -> -1.
    """
    try:
        v = int(v)
    except (TypeError, ValueError):
        return -1
    if v <= 0:
        return -1
    return v if v >= 10000 else v + 10000


def fetch_goes_flares() -> pd.DataFrame:
    """
    Returns DataFrame: [peak_time, start_time, goes_class, class_letter, peak_flux, noaa_ar].
    Keeps only M- and X-class events (peak flux >= 1e-5 W/m2), M and X in SEPARATE columns
    downstream — never merged.

    Catalog-cleaning fixes (applied BEFORE caching, so the cache is the clean catalog):
      * noaa_ar normalized to 5-digit form (1121 -> 11121)
      * dedup on (peak_time, goes_class); where raw duplicates disagreed on AR
        formatting, normalization makes them agree, and the row carrying a valid
        (non -1) AR is preferred as tie-break
      * 2018–2019 zero M/X rows: ACCEPTED as genuine deep solar minimum — do not
        backfill, do not re-query; test split (2018–2020) untouched
    """
    cache = os.path.join(OUT_DIR, "goes_flares_2010_2020.csv")
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["peak_time", "start_time"])
        log.info(f"GOES flares loaded from cache: {len(df)} events (already deduplicated)")
        return df

    from sunpy.net import Fido, attrs as a
    res = Fido.search(
        a.Time("2010-01-01", "2020-12-31"),
        a.hek.EventType("FL"),
        a.hek.FL.GOESCls >= "M1.0",
        a.hek.OBS.Observatory == "GOES",
    )
    tbl = res["hek"]

    def hek_time_to_str(col) -> list:
        """HEK time columns are astropy.time.Time — use .iso; fall back to str()."""
        try:
            return col.iso.tolist()
        except AttributeError:
            return [str(v) for v in col]

    df = pd.DataFrame({
        "peak_time": pd.to_datetime(hek_time_to_str(tbl["event_peaktime"])),
        "start_time": pd.to_datetime(hek_time_to_str(tbl["event_starttime"])),
        "goes_class": [str(c) for c in tbl["fl_goescls"]],
        "noaa_ar": [int(n) if str(n).isdigit() else -1 for n in tbl["ar_noaanum"]],
    })
    df["class_letter"] = df["goes_class"].str[0]
    mult = df["goes_class"].str[1:].replace("", "1.0").astype(float)
    df["peak_flux"] = np.where(df["class_letter"] == "X", 1e-4 * mult, 1e-5 * mult)
    df = df[df["class_letter"].isin(["M", "X"])].reset_index(drop=True)

    # ---------- FIX 1: AR normalization + dedup (BEFORE caching) ----------
    n_before = len(df)
    df["noaa_ar"] = df["noaa_ar"].map(_normalize_noaa_ar)
    # Prefer rows carrying a valid AR when duplicates disagree, then dedup on
    # the (peak_time, goes_class) physical-event key.
    df = (df.sort_values(["peak_time", "goes_class", "noaa_ar"],
                         ascending=[True, True, False])
            .drop_duplicates(subset=["peak_time", "goes_class"], keep="first")
            .sort_values("peak_time")
            .reset_index(drop=True))
    n_after = len(df)
    log.info(f"GOES dedup on (peak_time, goes_class): {n_before} -> {n_after} rows "
             f"({n_before - n_after} duplicate pipeline reports removed); "
             f"noaa_ar normalized to 5-digit form")

    # ---------- FIX 2: 2018–2019 gap — accepted, documented ----------
    for gap_year in (2018, 2019):
        n_gap = int((df["peak_time"].dt.year == gap_year).sum())
        log.info(f"GOES {gap_year}: {n_gap} M/X rows — "
                 "zero is EXPECTED (deep solar minimum); accepted as-is, no backfill")

    df.to_csv(cache, index=False)
    log.info(f"GOES flares fetched + cleaned + cached: {len(df)} M/X events")
    return df


flares = fetch_goes_flares()
print(flares["class_letter"].value_counts().to_string())
print(flares.head().to_string())

In [ ]:
# =====================================================================
# PHASE 7 / Step 7.2 — Fetch DONKI CME catalog (2010–2020) [REVISED]
# =====================================================================
def fetch_donki_cmes() -> pd.DataFrame:
    """
    Query NASA CCMC DONKI CMEAnalysis endpoint year-by-year.
    Returns DataFrame: [time, speed_kms, fast, source_location, lat, lon].

    Catalog notes (applied/verified BEFORE caching):
      * `sourceLocation` comes back EMPTY for every record in this period
        (~2,835 CME analyses) — this is the DONKI API itself, not an extraction
        bug. The column is retained for schema compatibility but must NOT be
        required downstream; use `lat`/`lon` for spatial stratification instead.
      * Slow CMEs (<= 500 km/s) are AMBIGUOUS, never negative (Step 7.2/7.3).
    """
    cache = os.path.join(OUT_DIR, "donki_cmes_2010_2020.csv")
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["time"])
        log.info(f"DONKI CMEs loaded from cache: {len(df)} entries")
        return df

    rows = []
    for y in range(2010, 2021):
        url = ("https://kauai.ccmc.gsfc.nasa.gov/DONKI/WS/get/CMEAnalysis"
               f"?startDate={y}-01-01&endDate={y}-12-31&mostAccurateOnly=true")
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            for item in (r.json() or []):
                spd = item.get("speed")
                t = item.get("time21_5")
                if spd is None or t is None:
                    continue
                rows.append({
                    "time": pd.to_datetime(t.replace("Z", "")),
                    "speed_kms": float(spd),
                    "source_location": item.get("sourceLocation", ""),
                    "lat": item.get("latitude"), "lon": item.get("longitude"),
                })
            log.info(f"DONKI {y}: {len(rows)} cumulative CME analyses")
        except Exception as e:
            log.error(f"DONKI {y} fetch failed: {e}")
    df = pd.DataFrame(rows)
    df["fast"] = df["speed_kms"] > CME_FAST_KMS

    # ---------- FIX: document empty source_location (BEFORE caching) ----------
    n_empty = int((df["source_location"].fillna("") == "").sum())
    log.info(f"DONKI source_location empty for {n_empty}/{len(df)} records "
             "(API-side for this period — EXPECTED; use lat/lon for spatial work, "
             "never require source_location)")

    df.to_csv(cache, index=False)
    return df


cmes = fetch_donki_cmes()
print(f"Total CME analyses: {len(cmes)} | fast (> {CME_FAST_KMS:.0f} km/s): {int(cmes['fast'].sum())}")
n_loc = int((cmes["source_location"].fillna("") != "").sum())
print(f"source_location populated: {n_loc}/{len(cmes)} (expected ~0 — use lat/lon instead)")

In [ ]:
# =====================================================================
# PHASE 7 / Step 7.4 — Confined-flare flag [REVISED: widened asymmetric window]
# =====================================================================
# The original +/-30 min window produced confined fractions of 0.935 (M) and
# 0.940 (X) against the ~0.4 literature expectation — an order-of-magnitude
# under-association. Root cause: DONKI `time21_5` marks coronagraph DETECTION
# at 21.5 R_sun, which lags true eruption onset by tens of minutes to ~2 h.
# Revised: asymmetric window  [peak - 30 min, peak + 120 min].
cme_times = pd.DatetimeIndex(cmes["time"]).sort_values()


def has_cme_association(peak_time: pd.Timestamp) -> bool:
    """True if any DONKI CME falls in [peak - 30 min, peak + 120 min]."""
    if len(cme_times) == 0:
        return False
    lo = cme_times.searchsorted(peak_time - CME_ASSOC_BEFORE, side="left")
    hi = cme_times.searchsorted(peak_time + CME_ASSOC_AFTER, side="right")
    return hi > lo


flares["confined_flag"] = [not has_cme_association(t) for t in flares["peak_time"]]
conf_rate = flares.groupby("class_letter")["confined_flag"].mean()
print(f"Association window: -{CME_ASSOC_BEFORE} / +{CME_ASSOC_AFTER} around flare peak")
print("Confined fraction by class (literature expectation ~0.4 for M; the old")
print("+/-30 min window gave 0.935/0.940 — should now be far closer to ~0.4):")
print(conf_rate.to_string())

In [ ]:
# =====================================================================
# PHASE 7 / Step 7.3 — Per-horizon label assignment
# =====================================================================
flare_M_times = pd.DatetimeIndex(flares.loc[flares.class_letter == "M", "peak_time"]).sort_values()
flare_X_times = pd.DatetimeIndex(flares.loc[flares.class_letter == "X", "peak_time"]).sort_values()
fast_cme_times = pd.DatetimeIndex(cmes.loc[cmes["fast"], "time"]).sort_values()
slow_cme_times = pd.DatetimeIndex(cmes.loc[~cmes["fast"], "time"]).sort_values()


def any_in(times: pd.DatetimeIndex, t0: pd.Timestamp, t1: pd.Timestamp) -> bool:
    """True if any event time falls in (t0, t1]."""
    lo = times.searchsorted(t0, side="right")
    hi = times.searchsorted(t1, side="right")
    return hi > lo


def label_window(t_end: pd.Timestamp) -> Dict[str, int]:
    """
    Labels for a window whose input ends at t_end, checked independently per horizon.
    M and X stay SEPARATE columns. cme_flag: 1 = fast CME, 0 = nothing in DONKI,
    -1 = only slow CMEs (ambiguous — excluded from the negative class in training).
    """
    out = {}
    for h in FORECAST_HORIZONS_H:
        t1 = t_end + pd.Timedelta(hours=h)
        out[f"flare_M_{h}h"] = int(any_in(flare_M_times, t_end, t1))
        out[f"flare_X_{h}h"] = int(any_in(flare_X_times, t_end, t1))
        if any_in(fast_cme_times, t_end, t1):
            out[f"cme_{h}h"] = 1
        elif any_in(slow_cme_times, t_end, t1):
            out[f"cme_{h}h"] = -1          # ambiguous — NOT negative
        else:
            out[f"cme_{h}h"] = 0
    return out


print("Label functions ready.")
print("Example labels @ 2017-09-06 09:00 (X9.3 day):",
      label_window(pd.Timestamp("2017-09-06 09:00:00")))

In [ ]:
# =====================================================================
# PHASE 8 / Steps 8.1–8.4 + PHASE 9 / Step 9.1 — Build the FULL window index
# =====================================================================
# The full index is cheap (metadata only, ~209,849 rows at stride 1/4/6). The
# REVISED sampling strategy filters which TRAIN rows get materialized — the
# index itself is built exactly as before so val/test remain untouched and the
# full-population statistics are still available for auditing.
def year_split(year: str) -> str:
    y = int(year)
    for name, (lo, hi) in SPLITS.items():
        if lo <= y <= hi:
            return name
    raise ValueError(year)


def build_window_index() -> pd.DataFrame:
    rows = []
    for yr, t in align.items():
        split = year_split(yr)
        stride = STRIDE[split]
        n = len(t)
        unusable = t["unusable"].values
        interp_gap = t["interp_ok"].values          # recoverable single-frame gaps
        kept = dropped = 0
        for s in range(0, n - T_IN + 1, stride):
            e = s + T_IN
            if unusable[s:e].any():                 # hard reject
                dropped += 1
                continue
            n_interp = int(interp_gap[s:e].sum())
            if n_interp > 1:                        # soft reject: > 1 gap
                dropped += 1
                continue
            t_end = t["time"].iloc[e - 1]
            rec = {"year": yr, "split": split, "row_start": s, "row_end": e,
                   "t_end": t_end, "n_interp": n_interp}
            rec.update(label_window(t_end))
            rows.append(rec)
            kept += 1
        log.info(f"{yr} [{split}]: windows kept={kept}, rejected={dropped}")
    return pd.DataFrame(rows)


window_index = build_window_index()
window_index.to_parquet(os.path.join(OUT_DIR, "window_index.parquet"))
print(f"\nTotal candidate windows: {len(window_index)} "
      "(reference full-construction figure: 209,849)")
if len(window_index):
    print(window_index.groupby("split").size().to_string())

## Phase 8-S (new) — Asymmetric stratified sampling of the TRAIN split

The original notebook materialized **all** stride-1 train windows (160,366 of ~209,849 total) at float32
(~1.5–2 TB). The revised strategy filters *which train rows get pixel-materialized*, before any pixel pull:

- **Positives** (any of the 12 horizon flags = 1 for M, X, or fast CME): kept at ~full density.
  Optional light stride-2 thinning of *M-only* clusters (`APPLY_M_THINNING`) — **never** applied to
  X-class or CME rows, which are the rarest and most valuable.
- **Ambiguous** rows (slow-CME `-1` flags, no positive): excluded from the negative pool entirely.
- **Negatives**: stratified by `(year, activity_bucket)`, `activity_bucket ∈ {quiet, active_subthreshold}`,
  derived from the **HMI-based PIL-length feature (Phase 5)** — *not* a new C-class GOES tag. This stops
  the negative class being ~95% featureless quiet disk, which would let the model shortcut on
  "active region present" instead of learning eruption dynamics.
- Target ratio **`TARGET_RATIO = 6` negatives per positive** (tunable notebook parameter).
- **Val and test are NOT class-conditionally subsampled** — they keep their stride-4/stride-6 continuous
  construction and true event prevalence, which is required for honest BSS / ECE / ROC-AUC evaluation later.

Because PIL length lives in pixel space, a cheap **activity proxy index** is built first: one HMI frame
sampled every `ACTIVITY_SAMPLE_EVERY = 30` alignment rows (≈ 6 h) per train year (~1,460 samples/year,
3 HMI channels each — a small, disk-cached, resumable pull), and each candidate window is assigned the
PIL length of its nearest-in-time sample.


In [ ]:
# =====================================================================
# PHASE 8-S / Step S.1 — Activity proxy index (PIL length @ ~6 h cadence)
# =====================================================================
def build_activity_index() -> pd.DataFrame:
    """
    Sparse PIL-length time series over the TRAIN years, one sample every
    ACTIVITY_SAMPLE_EVERY alignment rows (~6 h). Uses only the 3 HMI channels
    of a single 512x512 frame per sample — orders of magnitude cheaper than
    materialization. Disk-cached per run; resumable per year.
    """
    cache = os.path.join(OUT_DIR, "activity_index_train.parquet")
    if os.path.exists(cache):
        dfc = pd.read_parquet(cache)
        log.info(f"Activity index loaded from cache: {len(dfc)} samples")
        return dfc

    rows = []
    for yr in TRAIN_YEARS:
        t = align.get(yr)
        if t is None:
            continue
        vr = t[t["valid"]].iloc[::ACTIVITY_SAMPLE_EVERY]
        if ACTIVITY_MAX_PER_YEAR is not None:
            vr = vr.iloc[:ACTIVITY_MAX_PER_YEAR]
        for _, row in tqdm(vr.iterrows(), total=len(vr),
                           desc=f"activity proxy {yr}", leave=False):
            try:
                bz = apply_limb_mask(fetch_raw_frame(yr, "Bz", int(row["idx_Bz"])))
                bx = apply_limb_mask(fetch_raw_frame(yr, "Bx", int(row["idx_Bx"])))
                by = apply_limb_mask(fetch_raw_frame(yr, "By", int(row["idx_By"])))
                _, pil_len = compute_pil_mask(bx, by, bz)
                rows.append({"year": yr, "time": row["time"], "pil_length_px": pil_len})
            except Exception as e:
                log.warning(f"activity sample failed @ {row['time']}: {e}")
        gc.collect()
    dfa = pd.DataFrame(rows).sort_values("time").reset_index(drop=True)
    dfa.to_parquet(cache)
    log.info(f"Activity index built: {len(dfa)} samples across {TRAIN_YEARS}")
    return dfa


activity_index = build_activity_index()

# Bucket threshold: median PIL length over the sampled train frames. Below ->
# 'quiet' (featureless disk); above -> 'active_subthreshold' (AR present but no
# M/X/CME in the horizon). Two buckets per plan; threshold stored for the paper.
PIL_BUCKET_THRESH = float(activity_index["pil_length_px"].median()) if len(activity_index) else 0.0
print(f"activity_bucket threshold (median train PIL length): {PIL_BUCKET_THRESH:.0f} px")
print(activity_index["pil_length_px"].describe().to_string())

In [ ]:
# =====================================================================
# PHASE 8-S / Step S.2 — Stratified train filter (runs BEFORE materialization)
# =====================================================================
LABEL_COLS = ([f"flare_M_{h}h" for h in FORECAST_HORIZONS_H]
            + [f"flare_X_{h}h" for h in FORECAST_HORIZONS_H]
            + [f"cme_{h}h" for h in FORECAST_HORIZONS_H])


def _nearest_activity(times: pd.Series) -> np.ndarray:
    """PIL length of the nearest-in-time activity sample for each window t_end."""
    if len(activity_index) == 0:          # demo guard — everything 'quiet'
        return np.zeros(len(times))
    at = activity_index["time"].values
    av = activity_index["pil_length_px"].values
    pos = np.searchsorted(at, times.values)
    pos = np.clip(pos, 1, len(at) - 1)
    left_closer = (times.values - at[pos - 1]) <= (at[pos] - times.values)
    best = np.where(left_closer, pos - 1, pos)
    return av[best]


def filter_train_windows(idx: pd.DataFrame,
                         target_ratio: int = TARGET_RATIO,
                         seed: int = SAMPLING_SEED,
                         m_thinning: bool = APPLY_M_THINNING) -> pd.DataFrame:
    """
    Asymmetric, stratified selection of TRAIN windows to materialize.
    Val/test pass through COMPLETELY UNCHANGED (prevalence-preserving).
    """
    rng = np.random.default_rng(seed)
    train = idx[idx["split"] == "train"].copy()
    others = idx[idx["split"] != "train"].copy()

    lab = train[LABEL_COLS].values
    is_pos = (lab == 1).any(axis=1)
    cme_cols = [c for c in LABEL_COLS if c.startswith("cme_")]
    is_ambig = (~is_pos) & (train[cme_cols].values == -1).any(axis=1)
    is_neg = ~is_pos & ~is_ambig

    # ---- Positives: full density; optional stride-2 thinning of M-ONLY rows ----
    pos = train[is_pos].copy()
    if m_thinning:
        x_cols = [c for c in LABEL_COLS if c.startswith("flare_X_")]
        m_cols = [c for c in LABEL_COLS if c.startswith("flare_M_")]
        m_only = ((pos[m_cols].values == 1).any(axis=1)
                  & ~(pos[x_cols].values == 1).any(axis=1)
                  & ~(pos[cme_cols].values == 1).any(axis=1))
        keep = np.ones(len(pos), dtype=bool)
        m_idx = np.where(m_only)[0]
        keep[m_idx[1::2]] = False               # stride-2 within M-only cluster order
        pos = pos[keep]
        log.info(f"M-only thinning: removed {int((~keep).sum())} rows "
                 "(X-class and CME positives NEVER thinned)")

    # ---- Negatives: stratify by (year, activity_bucket) ----
    neg = train[is_neg].copy()
    neg["pil_proxy"] = _nearest_activity(neg["t_end"])
    neg["activity_bucket"] = np.where(neg["pil_proxy"] >= PIL_BUCKET_THRESH,
                                      "active_subthreshold", "quiet")

    quota_total = target_ratio * len(pos)
    years = sorted(neg["year"].unique())
    buckets = ["quiet", "active_subthreshold"]
    per_cell = max(quota_total // (len(years) * len(buckets)), 1)

    picks, shortfall = [], 0
    for yr in years:
        for b in buckets:
            cell = neg[(neg["year"] == yr) & (neg["activity_bucket"] == b)]
            take = min(per_cell, len(cell))
            shortfall += per_cell - take
            if take:
                picks.append(cell.sample(n=take, random_state=int(rng.integers(1 << 31))))
    # Redistribute any shortfall uniformly over the remaining pool
    chosen = pd.concat(picks) if picks else neg.iloc[:0]
    if shortfall > 0:
        remaining = neg.drop(chosen.index)
        extra = min(shortfall, len(remaining))
        if extra:
            chosen = pd.concat([chosen, remaining.sample(
                n=extra, random_state=int(rng.integers(1 << 31)))])

    train_filtered = pd.concat([pos, chosen]).sort_values("t_end").reset_index(drop=True)

    print("=== Train filter summary ===")
    print(f"  candidate train windows : {len(train)}")
    print(f"  positives kept          : {len(pos)}")
    print(f"  ambiguous excluded      : {int(is_ambig.sum())} (slow-CME -1, not negative)")
    print(f"  negatives sampled       : {len(chosen)} / {int(is_neg.sum())} "
          f"(target ratio {target_ratio}:1)")
    print(f"  materialized train total: {len(train_filtered)} "
          f"(expected ~29,800: ~8,000 pos + ~21,800 stratified neg)")
    if "activity_bucket" in chosen.columns and len(chosen):
        print("  negative stratification (year x bucket):")
        print(chosen.groupby(["year", "activity_bucket"]).size().unstack(fill_value=0).to_string())
    return pd.concat([train_filtered, others], ignore_index=True)


window_index_mat = filter_train_windows(window_index)
window_index_mat.to_parquet(os.path.join(OUT_DIR, "window_index_materialized.parquet"))
print("\n=== Windows to MATERIALIZE, per split ===")
print(window_index_mat.groupby("split").size().to_string())
print("(val/test counts identical to the full index — prevalence preserved)")

In [ ]:
# =====================================================================
# PHASE 8 — Materialise window tensors [REVISED: float16 storage]
# =====================================================================
# Each shard = SHARD_SIZE windows saved as one compressed .npz:
#   X        (N, 8, 10, 256, 256) float16  - normalised input tensors  [REVISED]
#   scalars  (N, 8, 3)            float16  - [pil_length, shear/90, flux_rate_z] [REVISED]
#   labels   (N, 12)              int8     - [M,X,CME] x [6,12,24,48]h
#   t_end    (N,)                 datetime64
#
# *** float16 is a STORAGE format only. The training notebook MUST upcast to
# float32 (or use a Keras mixed-precision policy) immediately after loading —
# NLL / logvar loss terms are numerically unstable computed directly in fp16. ***
SHARD_SIZE = 64

active_scalers = scalers   # or scalers_surya when USE_SURYA_SCALERS


def norm_scalar(name: str, v: float) -> float:
    s = active_scalers[name]
    if s["kind"] == "div":
        return v / s["div"]
    lo = s["mean"] - s["clip"] * s["std"]; hi = s["mean"] + s["clip"] * s["std"]
    return (np.clip(v, lo, hi) - s["mean"]) / s["std"]


def materialise_window(rec: pd.Series) -> Tuple[np.ndarray, np.ndarray]:
    """Assemble + normalise one window. Returns (X[8,10,256,256], scalars[8,3]).

    Every raw frame the window needs is pulled through the lazy Dask graph in
    ONE compute per channel (overlapping zarr chunks read once)."""
    yr, t = rec["year"], align[rec["year"]]
    seq = list(range(rec["row_start"], rec["row_end"]))

    plan = []                                   # ("direct", r) | ("interp", r-1, r+1)
    need = {ch: set() for ch in RAW_CHANNELS}
    for r in seq:
        if t.iloc[r]["valid"]:
            plan.append(("direct", r))
            src_rows = (r,)
        else:
            plan.append(("interp", r - 1, r + 1))
            src_rows = (r - 1, r + 1)
        for rr in src_rows:
            for ch in RAW_CHANNELS:
                need[ch].add(int(t.iloc[rr][f"idx_{ch}"]))

    cleaned: Dict[str, Dict[int, np.ndarray]] = {ch: {} for ch in RAW_CHANNELS}
    cr_frac: Dict[str, Dict[int, float]] = {ch: {} for ch in RAW_CHANNELS}
    for ch in RAW_CHANNELS:
        idxs = sorted(i for i in need[ch] if i >= 0)
        frames = fetch_raw_frames(yr, ch, idxs)
        for k, raw_i in enumerate(idxs):
            fr, frac = _clean_and_correct(yr, ch, frames[k])
            cleaned[ch][raw_i] = fr
            cr_frac[ch][raw_i] = frac

    def stack_for_row(r: int) -> Tuple[np.ndarray, Dict[str, float]]:
        c512 = {ch: cleaned[ch][int(t.iloc[r][f"idx_{ch}"])] for ch in RAW_CHANNELS}
        crf = {ch: cr_frac[ch][int(t.iloc[r][f"idx_{ch}"])] for ch in RAW_CHANNELS}
        return assemble_from_cleaned(yr, c512, crf)

    stacks, scals, prev_flux = [], [], None
    for item in plan:
        if item[0] == "direct":
            stack, sc = stack_for_row(item[1])
        else:
            s_prev, _ = stack_for_row(item[1])
            s_next, sc = stack_for_row(item[2])
            stack = 0.5 * (s_prev + s_next)
        flux_rate = (0.0 if prev_flux is None
                     else (sc["unsigned_flux_Mx"] - prev_flux) / (HMI_CADENCE_MIN * 60))
        prev_flux = sc["unsigned_flux_Mx"]
        stacks.append(normalise_stack(stack, active_scalers))
        scals.append([sc["pil_length_px"],
                      norm_scalar("shear_angle", sc["shear_angle_deg"]),
                      norm_scalar("flux_rate", flux_rate)])
    return np.stack(stacks), np.array(scals, dtype=np.float32)


def write_shards(split: str, max_shards: Optional[int] = None):
    """Materialise the FILTERED window set of a split into resumable .npz shards.
    Skip-if-exists per shard => interruptible & restartable."""
    sub = window_index_mat[window_index_mat["split"] == split].reset_index(drop=True)
    shard_dir = os.path.join(OUT_DIR, split)
    os.makedirs(shard_dir, exist_ok=True)
    n_shards = int(np.ceil(len(sub) / SHARD_SIZE))
    for si in range(n_shards):
        if max_shards is not None and si >= max_shards:
            log.info(f"{split}: stopping after {max_shards} shard(s) (demo limit)")
            break
        path = os.path.join(shard_dir, f"shard_{si:05d}.npz")
        if os.path.exists(path):
            continue                                          # resumable
        chunk = sub.iloc[si * SHARD_SIZE:(si + 1) * SHARD_SIZE]
        Xs, Ss, Ls, Ts = [], [], [], []
        for _, rec in tqdm(chunk.iterrows(), total=len(chunk),
                           desc=f"{split} shard {si}", leave=False):
            try:
                X, S = materialise_window(rec)
            except Exception as e:
                log.warning(f"window @ {rec['t_end']} failed: {e}")
                continue
            Xs.append(X.astype(STORE_DTYPE))                  # float16 STORAGE
            Ss.append(S.astype(STORE_DTYPE))
            Ls.append(rec[LABEL_COLS].values.astype(np.int8))
            Ts.append(np.datetime64(rec["t_end"]))
        if Xs:
            np.savez_compressed(path, X=np.stack(Xs), scalars=np.stack(Ss),
                                labels=np.stack(Ls), t_end=np.array(Ts))
            log.info(f"wrote {path} ({len(Xs)} windows, dtype={STORE_DTYPE.__name__})")
        del Xs, Ss, Ls, Ts
        gc.collect()                                          # keep RAM flat


# DEMO: 1 shard per split validates the pipeline end-to-end.
# PRODUCTION: write_shards(split, max_shards=None) — long-running, resumable.
for split_name in ["train", "val", "test"]:
    write_shards(split_name, max_shards=1)

In [ ]:
# =====================================================================
# PHASE 9 / Steps 9.2–9.4 — Leakage checks & class-rate documentation
# =====================================================================

# Step 9.2 — Scaler-leakage check UNCHANGED: scalers fit only on 2010–2014
# frames; Surya scalers never refit. Report val/test shift under TRAIN scalers.
def normalisation_shift_report(sample_per_year: int = 10):
    rows = []
    for split in ["val", "test"]:
        yrs = [str(y) for y in range(SPLITS[split][0], SPLITS[split][1] + 1)]
        for ch in CHANNEL_ORDER:
            vals = []
            for yr in yrs:
                t = align.get(yr)
                if t is None:
                    continue
                vr = t[t["valid"]]
                pick = vr.iloc[np.linspace(0, len(vr) - 1,
                                           min(sample_per_year, len(vr)), dtype=int)]
                for _, row in pick.iterrows():
                    try:
                        stack, _ = assemble_timestep(yr, row)
                        x = normalise_stack(stack, active_scalers)[CHANNEL_ORDER.index(ch)]
                        vals.append([x.mean(), x.std()])
                    except Exception:
                        continue
            if vals:
                v = np.array(vals)
                rows.append({"split": split, "channel": ch,
                             "mean_shift": round(float(v[:, 0].mean()), 3),
                             "std_ratio": round(float(v[:, 1].mean()), 3)})
    return pd.DataFrame(rows)


RUN_SHIFT_CHECK = False    # network-heavy; enable for production audit
if RUN_SHIFT_CHECK:
    shift_df = normalisation_shift_report()
    print("Normalised mean should be ~0, std ~1. Larger shifts expected for "
          "activity-sensitive channels (131A) near solar minimum:")
    print(shift_df.to_string())

print("POLICY: hyperparameter tuning on VAL only; TEST touched once per model.")
print("POLICY: scalers fit on 2010–2014 ONLY; Surya scalers are NEVER refit.\n")

# Step 9.4 — class rates. REVISED reporting: filtered TRAIN rates are reported
# SEPARATELY from the untouched val/test rates — train rates differ BY DESIGN
# (stratified 6:1), while val/test must still match the original expectation
# (~1–3% M / 0.1–0.3% X / 0.5–1.5% fast CME of 24h windows).
def class_rate_table(idx: pd.DataFrame, tag: str) -> pd.DataFrame:
    rows = []
    for split in ["train", "val", "test"]:
        sub = idx[idx["split"] == split]
        rows.append({
            "set": tag, "split": split, "windows": len(sub),
            "M_rate_24h_%": round(100 * (sub["flare_M_24h"] == 1).mean(), 3),
            "X_rate_24h_%": round(100 * (sub["flare_X_24h"] == 1).mean(), 3),
            "fastCME_rate_24h_%": round(100 * (sub["cme_24h"] == 1).mean(), 3),
            "ambiguousCME_24h_%": round(100 * (sub["cme_24h"] == -1).mean(), 3),
        })
    return pd.DataFrame(rows)


class_rates = pd.concat([
    class_rate_table(window_index, "full_index"),
    class_rate_table(window_index_mat, "materialized"),
]).set_index(["set", "split"])
print(class_rates.to_string())
print("\nNOTE: materialized-train rates are INTENTIONALLY elevated (stratified "
      "6:1 rebalancing). Val/test rows are identical between the two tables — "
      "these preserve true prevalence for honest BSS / ECE / ROC-AUC evaluation.")
class_rates.to_csv(os.path.join(OUT_DIR, "class_rates_per_split.csv"))

In [ ]:
# =====================================================================
# PHASE 10 / Step 10.1 — Label sanity on known major events [RETAINED]
# =====================================================================
KNOWN_EVENTS = [
    ("2017-09-06 11:53", "X", "X9.3 — 6 Sep 2017"),
    ("2017-09-10 16:06", "X", "X8.2 — 10 Sep 2017"),
    ("2011-02-15 01:56", "X", "X2.2 — 15 Feb 2011 (first X of Cycle 24)"),
]

for t_str, cls, name in KNOWN_EVENTS:
    t_evt = pd.Timestamp(t_str)
    lab = label_window(t_evt - pd.Timedelta(hours=3))
    ok = lab[f"flare_{cls}_6h"] == 1
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: window@t-3h -> flare_{cls}_6h = {lab[f'flare_{cls}_6h']}")

# The X9.3 / X8.2 windows fall in 2017 => VAL split, which is untouched by the
# sampling filter — confirm they are present in the materialization index too.
for t_str, cls, name in KNOWN_EVENTS[:2]:
    t_evt = pd.Timestamp(t_str)
    near = window_index_mat[
        (window_index_mat["t_end"] > t_evt - pd.Timedelta(hours=6))
        & (window_index_mat["t_end"] < t_evt)
        & (window_index_mat[f"flare_{cls}_6h"] == 1)]
    print(f"[{'PASS' if len(near) else 'FAIL'}] {name}: "
          f"{len(near)} correctly-flagged windows in materialization index")

pos = window_index_mat[window_index_mat["flare_M_24h"] == 1]
if len(pos) > 0:
    spot = pos.sample(min(5, len(pos)), random_state=42)
    print(f"\nSpot-check sample ({len(spot)} positive M-flare windows):")
    print(spot[["year", "split", "t_end", "flare_M_24h", "flare_X_24h", "cme_24h"]].to_string())

In [ ]:
# =====================================================================
# PHASE 10 / Steps 10.2 + 10.3 — Balance audit & autocorrelation [on FILTERED set]
# =====================================================================
# Step 10.2 — rerun against the filtered materialization set. Train differs by
# design; val/test must still sit at ~1–3% M / 0.1–0.3% X / 0.5–1.5% CME.
for split in ["train", "val", "test"]:
    sub = window_index_mat[window_index_mat["split"] == split]
    tag = "(stratified — intentionally elevated)" if split == "train" else "(must match original expectation)"
    print(f"{split:>5} {tag}: "
          f"M={100*(sub['flare_M_24h']==1).mean():.2f}%  "
          f"X={100*(sub['flare_X_24h']==1).mean():.3f}%  "
          f"CME={100*(sub['cme_24h']==1).mean():.2f}%")

# Step 10.3 — temporal autocorrelation of the FILTERED training labels.
train_lbl = window_index_mat[window_index_mat["split"] == "train"].sort_values("t_end")
y = (train_lbl["flare_M_24h"] == 1).astype(float).values
if len(y) > 500:
    y = y - y.mean()
    max_lag = 400
    ac = np.array([np.corrcoef(y[:-k], y[k:])[0, 1] if k > 0 else 1.0
                   for k in range(max_lag)])
    plt.figure(figsize=(10, 3.5))
    plt.plot(ac, lw=1)
    plt.axvline(200, color="r", ls="--", label="lag 200 — should be ~0 beyond")
    plt.axhline(0, color="k", lw=0.5)
    plt.title("Autocorrelation of flare_M_24h label (FILTERED train)")
    plt.xlabel("lag (windows)"); plt.ylabel("ACF"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"ACF @ lag 200: {ac[200]:.4f} (stratified subsampling breaks contiguity, "
          "so decay should be FASTER than the stride-1 original)")
else:
    print("Not enough train windows for the autocorrelation check (demo run).")

In [ ]:
# =====================================================================
# PHASE 10 / Step 10.5 — Visual verification of Jz and PIL maps [RETAINED]
# =====================================================================
def verify_jz_pil(year: str, when: str, title: str):
    t = align.get(year)
    if t is None or not t["valid"].any():
        print(f"{year}: no valid timestamps"); return
    target = pd.Timestamp(when)
    vr = t[t["valid"]]
    row = vr.iloc[(vr["time"] - target).abs().argmin()]
    bz = apply_limb_mask(fetch_raw_frame(year, "Bz", int(row["idx_Bz"])))
    bx = apply_limb_mask(fetch_raw_frame(year, "Bx", int(row["idx_Bx"])))
    by = apply_limb_mask(fetch_raw_frame(year, "By", int(row["idx_By"])))
    jz = compute_jz(bx, by)
    pil, plen = compute_pil_mask(bx, by, bz)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    v = np.nanpercentile(np.abs(bz), 99) or 1
    axes[0].imshow(bz, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
    axes[0].set_title(f"HMI Bz — {row['time']}")
    jv = np.nanpercentile(np.abs(jz), 99) or 1
    axes[1].imshow(jz, origin="lower", cmap="PuOr_r", vmin=-jv, vmax=jv)
    axes[1].set_title("Jz (vertical current density)")
    axes[2].imshow(bz, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
    axes[2].contour(pil, levels=[0.5], colors="lime", linewidths=0.8)
    axes[2].set_title(f"PIL mask overlay (length = {plen}px)")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()


verify_jz_pil("2017", "2017-09-06 09:00", "ACTIVE — AR 12673, X9.3 flare day")
verify_jz_pil("2019", "2019-06-15 12:00", "QUIET — solar minimum")

In [ ]:
# =====================================================================
# SUMMARY — Expected vs. actual output sizes  +  production checklist
# =====================================================================
expected = pd.DataFrame([
    {"split": "train", "windows_old_stride146": 160366,
     "windows_new_strategy": "~29,800 (~8,000 pos + ~21,800 stratified neg)"},
    {"split": "val",   "windows_old_stride146": 29448,
     "windows_new_strategy": "29,448 (unchanged — prevalence-preserving)"},
    {"split": "test",  "windows_old_stride146": 20035,
     "windows_new_strategy": "20,035 (unchanged — prevalence-preserving)"},
    {"split": "TOTAL", "windows_old_stride146": 209849,
     "windows_new_strategy": "~79,300"},
]).set_index("split")
print("=== EXPECTED WINDOW COUNTS ===")
print(expected.to_string())

storage = pd.DataFrame([
    {"format": "float32 (old, full stride-1 train)", "est_compressed": "~1.5–2 TB"},
    {"format": "float32, new window selection",       "est_compressed": "~790 GB"},
    {"format": "float16, new window selection (THIS RUN)", "est_compressed": "~395 GB"},
]).set_index("format")
print("\n=== EXPECTED STORAGE ===")
print(storage.to_string())

# ---- Confirm expected counts against the materialization index + written shards ----
print("\n=== ACTUAL (this run) ===")
idx_counts = window_index_mat.groupby("split").size()
total_bytes = 0
for split in ["train", "val", "test"]:
    shard_dir = os.path.join(OUT_DIR, split)
    files = sorted(f for f in os.listdir(shard_dir)) if os.path.isdir(shard_dir) else []
    n_win, n_bytes = 0, 0
    for f in files:
        p = os.path.join(shard_dir, f)
        n_bytes += os.path.getsize(p)
        try:
            with np.load(p) as z:
                n_win += z["labels"].shape[0]
                assert z["X"].dtype == STORE_DTYPE, f"{f}: X dtype {z['X'].dtype} != {STORE_DTYPE}"
        except Exception as e:
            log.warning(f"shard scan {f}: {e}")
    total_bytes += n_bytes
    print(f"{split:>5}: index={idx_counts.get(split, 0):>6} windows | "
          f"materialized so far={n_win:>6} in {len(files)} shard(s) | "
          f"{n_bytes/1e9:.2f} GB on disk")
print(f"TOTAL on disk so far: {total_bytes/1e9:.2f} GB "
      f"(full production target ~395 GB at float16)")

print("""
=== PRODUCTION RUN CHECKLIST ===
1. RUN_DISAMBIG_SCAN = True        — full HMI disambiguation jump scan (Step 4.3)
2. RUN_SHIFT_CHECK = True          — normalisation shift audit (Step 9.2)
3. write_shards(split, max_shards=None) for all three splits — resumable
4. Optionally USE_SURYA_SCALERS = True for Model D / F input variants
5. TARGET_RATIO / APPLY_M_THINNING are the tunable sampling knobs (Phase 8-S)

Artefacts in ./sdoml_dataset_out/:
  coverage_stats.csv · scalers.json (+ scalers_surya.json) ·
  goes_flares_2010_2020.csv (deduplicated) · donki_cmes_2010_2020.csv ·
  window_index.parquet (full) · window_index_materialized.parquet (filtered) ·
  activity_index_train.parquet · class_rates_per_split.csv ·
  train/ val/ test/  — float16 .npz shards:
    X (N,8,10,256,256) float16 · scalars (N,8,3) float16 ·
    labels (N,12) int8 · t_end (N,) datetime64
REMINDER: training must upcast X/scalars to float32 on load (fp16 NLL is unstable).
""")